# Invoice intake: evaluation notebook

Checks the pieces of the pipeline on the sample invoices in `tests/samples`:
1. field-level extraction accuracy
2. OpenCV de-skew on a rotated page
3. the agent's validate-and-retry trace

Run from the `python-service` folder (`jupyter nbconvert --execute` or open it in Jupyter).

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, cv2, pandas as pd
from datetime import date
from app.extract import RuleBasedExtractor, FIELDS
from app.agent import InvoiceAgent
from app.validator import LocalValidator
from app.preprocess import estimate_skew, preprocess
SAMPLES = Path.cwd().parent / 'tests' / 'samples'
expected = json.loads((SAMPLES / 'expected.json').read_text(encoding='utf-8'))
len(expected), 'sample invoices'

(5, 'sample invoices')

## 1. Field-level accuracy (rule-based extractor)

In [2]:
rows = []
ex = RuleBasedExtractor()
for name, exp in expected.items():
    got = ex.extract(text=(SAMPLES / name).read_text(encoding='utf-8'))
    for f in FIELDS:
        rows.append({'invoice': name, 'field': f, 'expected': exp[f], 'got': got[f], 'ok': exp[f] == got[f]})
df = pd.DataFrame(rows)
print(f"overall: {df.ok.mean():.0%} of {len(df)} fields correct")
df.groupby('field').ok.mean().round(2).to_frame('accuracy')

overall: 100% of 40 fields correct


,accuracy
field,
currency,1.0
gstin,1.0
invoiceDate,1.0
invoiceNumber,1.0
subtotal,1.0
taxAmount,1.0
total,1.0
vendor,1.0


In [3]:
df[~df.ok]  # empty means every field matched

,invoice,field,expected,got,ok


## 2. OpenCV de-skew
A page of text is rotated by a known angle; `preprocess` should straighten it.

In [4]:
def page(angle):
    img = np.full((800, 600), 255, np.uint8)
    for i in range(10):
        cv2.putText(img, f'Invoice line number {i} total 1,180.00', (40, 80 + i * 65),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, 0, 2, cv2.LINE_AA)
    m = cv2.getRotationMatrix2D((300, 400), angle, 1.0)
    return cv2.warpAffine(img, m, (600, 800), borderValue=255)
out = []
for angle in (-8, -4, 0, 3, 6):
    src = page(angle)
    fixed = cv2.imdecode(np.frombuffer(preprocess(cv2.imencode('.png', src)[1].tobytes()), np.uint8), cv2.IMREAD_GRAYSCALE)
    out.append({'rotated_by': angle, 'skew_before': round(estimate_skew(src), 2), 'skew_after': round(estimate_skew(fixed), 2)})
pd.DataFrame(out)

,rotated_by,skew_before,skew_after
0,-8,-7.46,0.00
1,-4,-3.57,0.00
2,0,0.00,0.00
3,3,3.49,-0.23
4,6,6.22,0.00


## 3. Agent trace on an inconsistent invoice
The total does not equal subtotal + tax, so the agent reports it for review instead of guessing.

In [5]:
agent = InvoiceAgent(RuleBasedExtractor(), LocalValidator(date(2026, 10, 19)), max_attempts=3)
res = agent.run(text=(SAMPLES / 'total_mismatch.txt').read_text(encoding='utf-8'))
print(res.status, '| attempts:', res.attempts, '| errors:', res.errors)
pd.DataFrame([{'attempt': t['attempt'], 'feedback': t['feedback'], 'errors': t['errors']} for t in res.trace])

needs_review | attempts: 2 | errors: ['total must equal subtotal + taxAmount']


,attempt,feedback,errors
0,1,None,[total must equal subtotal + taxAmount]
1,2,[total must equal subtotal + taxAmount],[total must equal subtotal + taxAmount]


## Limits
* The rule-based extractor only reads text invoices; scanned images go to the language-model extractor (`INTAKE_EXTRACTOR=llm`), which is not scored here because it needs an API key.
* Five sample invoices is a smoke test, not a benchmark. Add real (anonymised) invoices to `tests/samples` to extend it.